# 01b — Pembangunan Ulang Dataset (satu pipeline, dapat diulang dari data mentah)

**Masalah di notebook 01 lama (terverifikasi dari berkasnya):**
1. ±23 sel menimpa `dataset_politik_balanced.csv` di tempat; nomor eksekusi tidak berurutan → tidak bisa direproduksi dengan *Run All*.
2. Panjang dipotong berbeda per label (fakta ≤45 kata, hoaks ≤80 kata) → panjang menjadi petunjuk label.
3. `f"{title}. {body}"` menggandakan judul pada fakta (99,8% vs 5,4% pada hoaks).
4. Berkas Komdigi tidak punya kolom isi (`body_text`/`excerpt`), tetapi kode lama tetap memakainya.
5. Sub-topik ditentukan dengan pencocokan *substring* (mis. `"mk"` cocok di `"pemkot"`).
6. Near-duplicate hanya dicek 80 karakter pertama; berita yang sama dari portal berbeda bisa terpisah ke train dan test.
7. Penyeimbangan hanya menyamakan jumlah per label, bukan topik/panjang.

**Prinsip notebook ini:** satu aturan untuk SEMUA kelas, tidak ada penimpaan berkas, setiap filter dicatat per label dan sumber,
seed tetap, dan *kriteria penerimaan ditetapkan di bawah SEBELUM melihat hasil*.

**Keterbatasan yang tidak bisa diselesaikan kode:** fakta semuanya dari portal berita, hoaks semuanya dari situs pemeriksa fakta.
Sumber masih sempurna berkorelasi dengan label. Notebook ini hanya mengukurnya dan menyiapkan slot `EXTRA_SOURCES` untuk sumber tambahan.

### Kriteria penerimaan (ditetapkan sebelum eksekusi; ubah hanya dengan alasan tertulis)
| # | Kriteria | Ambang |
|---|---|---|
| C1 | Selisih tingkat judul ganda antar label, tiap split | ≤ 5 poin |
| C2 | KS-statistic panjang kata antar label | ≤ 0,10 |
| C3 | Selisih tingkat teks berakhir tanda baca antar label | ≤ 5 poin |
| C4 | Akurasi pohon keputusan memakai fitur panjang saja (test) | ≤ 0,60 |
| C5 | Tidak ada group (klaster near-duplicate) yang melintasi split | 0 pelanggaran |
| C6 | Selisih proporsi sub-topik antar label | ≤ 2 poin |
| INFO | Akurasi 8-kata-pertama (TF-IDF+LR), ketimpangan sumber, tumpang-tindih tahun | dilaporkan, tanpa ambang |

# 1

In [1]:
import os, sys, re, json, hashlib, platform, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import ks_2samp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score

# =============================================================================
# PELACAK OTOMATIS text_pipeline.py (KEBAL LOKASI DIREKTORI KERJA)
# =============================================================================
current_dir = Path(os.getcwd()).resolve()
found_path = None

# Cek di direktori saat ini, satu tingkat di atas (root), atau subdirektori
search_dirs = [current_dir, current_dir.parent, current_dir / "notebooks", current_dir / "src"]
for d in search_dirs:
    candidate = d / "text_pipeline.py"
    if candidate.is_file():
        found_path = d
        break

# Jika belum ketemu, lakukan pencarian rekursif di seluruh folder proyek
if not found_path:
    for p in current_dir.parent.rglob("text_pipeline.py"):
        found_path = p.parent
        break

if found_path:
    if str(found_path) not in sys.path:
        sys.path.insert(0, str(found_path))
    print(f"✓ Berhasil menemukan text_pipeline.py di: {found_path}")
else:
    # Cek apakah terjebak ekstensi .txt di Windows
    txt_candidates = list(current_dir.parent.rglob("text_pipeline.py.txt"))
    if txt_candidates:
        raise FileNotFoundError(
            f"❌ File terdeteksi bernama '{txt_candidates[0].name}'! "
            f"Windows menambahkan ekstensi .txt. Silakan rename file tersebut menjadi 'text_pipeline.py'."
        )
    raise FileNotFoundError(
        "❌ File 'text_pipeline.py' sama sekali tidak ditemukan di root maupun folder notebooks! "
        "Pastikan file sudah disimpan di folder proyek."
    )

# Impor modul pipeline
import text_pipeline as tp
print("✓ Berhasil import text_pipeline as tp!")

# =============================================================================
# KONFIGURASI DATASET
# =============================================================================
SEED = 42
RAW_DIR = os.environ.get("RAW_DIR") or ("../data/raw" if os.path.exists("../data/raw") else "data/raw")
OUT_DIR = os.environ.get("OUT_DIR") or ("../data/processed_v2" if os.path.exists("../data") else "data/processed_v2")
os.makedirs(OUT_DIR, exist_ok=True)

CFG = dict(
    MIN_WORDS=10, 
    MAX_WORDS=60, 
    NEAR_DUP_COS=0.90, 
    GROUP_COS=0.70,
    LEN_BINS=[0, 20, 30, 40, 50, 60, 10_000], 
    N_FOLDS=20, 
    TRAIN_FOLDS=14, 
    VAL_FOLDS=2, 
    DEVTEST_FOLDS=2, 
    LOCKBOX_FOLDS=2, 
    SEED=SEED
)

SOURCES = [
    dict(name="turnbackhoax.id",   label=1, file="dataset_turnbackhoax_10_cleaned", title=["Title"], body=["Clean Narasi", "Narasi"], date=["Date", "Tanggal", "date"]),
    dict(name="komdigi.go.id",     label=1, file="komdigi_hoaks",                   title=["title"], body=["body_text", "excerpt"],   date=["published_at"]),
    dict(name="hoax_archive_15k",  label=1, file="hoax_dataset_with_text",          title=["judul"], body=["teks", "deskripsi"],     date=["tanggal"]),
    dict(name="cnnindonesia.com",  label=0, file="dataset_cnn_10k_cleaned",         title=["Title"], body=["text_new", "FullText"],  date=["Date", "date", "Tanggal"]),
    dict(name="tempo.co",          label=0, file="dataset_tempo_6k_cleaned",        title=["Title"], body=["text_new", "FullText"],  date=["Date", "date", "Tanggal"]),
    dict(name="kompas.com",        label=0, file="dataset_kompas_4k_cleaned",       title=["Title"], body=["text_new", "FullText"],  date=["Date", "date", "Tanggal"]),
]
EXTRA_SOURCES = []
USE_TITLE_ONLY_SOURCES = False

LOG = []
def log(step, df, note=""):
    g = df.groupby(["source", "label"]).size()
    for (s, l), n in g.items():
        LOG.append(dict(step=step, source=s, label=int(l), n=int(n), note=note))
    print(f"[{step:<28}] total={len(df):>7,} | per label={df.label.value_counts().sort_index().to_dict()}  {note}")

✓ Berhasil menemukan text_pipeline.py di: E:\SEMESTER 5\TA\PROJECT\notebooks
✓ Berhasil import text_pipeline as tp!


# Tahap 2: Eksekusi Pembangunan Ulang Dataset Bersih

In [2]:
# ---------------- 1. MUAT DATA MENTAH (tanpa menimpa apa pun) ----------------
def pick(row, names):
    for n in names:
        if n in row.index:
            v = row[n]
            if pd.notna(v) and str(v).strip() not in ("", "nan", "None"):
                return str(v).strip()
    return ""

def load_source(s):
    files = sorted(f for f in os.listdir(RAW_DIR) if f.startswith(s["file"] + ".")) or \
            sorted(f for f in os.listdir(RAW_DIR) if f.startswith(s["file"]))
    if not files:
        print(f"!! BERKAS TIDAK DITEMUKAN untuk {s['name']}: {s['file']}"); return pd.DataFrame()
    p = os.path.join(RAW_DIR, files[0])
    d = pd.read_excel(p) if p.endswith((".xlsx", ".xls")) else pd.read_csv(p)
    out = pd.DataFrame({
        "title":    [pick(r, s["title"]) for _, r in d.iterrows()],
        "body":     [pick(r, s["body"])  for _, r in d.iterrows()],
        "date_raw": [pick(r, s["date"])  for _, r in d.iterrows()],
    })
    out["date"] = pd.to_datetime(out["date_raw"], errors="coerce")
    out["source"], out["label"], out["file"] = s["name"], s["label"], files[0]
    print(f"{s['name']:<18} {files[0]:<45} baris={len(out):>6,} | isi kosong={ (out.body=='').mean()*100:5.1f}% | tanggal terbaca={out.date.notna().mean()*100:5.1f}%")
    return out

raw = pd.concat([load_source(s) for s in SOURCES + EXTRA_SOURCES], ignore_index=True)
log("1_raw", raw)

no_body = raw.assign(e=(raw.body == "")).groupby("source").e.mean()
title_only = no_body[no_body > 0.95].index.tolist()
if title_only and not USE_TITLE_ONLY_SOURCES:
    print(f"\n>> Sumber tanpa isi dikeluarkan (USE_TITLE_ONLY_SOURCES=False): {title_only}")
    raw = raw[~raw.source.isin(title_only)].reset_index(drop=True)
    log("1b_tanpa_sumber_judul_saja", raw)

turnbackhoax.id    dataset_turnbackhoax_10_cleaned.csv           baris=10,381 | isi kosong= 37.3% | tanggal terbaca=  0.0%
komdigi.go.id      komdigi_hoaks.csv.csv                         baris= 3,624 | isi kosong=  0.0% | tanggal terbaca=100.0%


C:\Users\ASUS\AppData\Local\Temp\ipykernel_2676\206872903.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["date"] = pd.to_datetime(out["date_raw"], errors="coerce")


hoax_archive_15k   hoax_dataset_with_text (2).csv.csv            baris=15,720 | isi kosong=  0.0% | tanggal terbaca= 24.1%
cnnindonesia.com   dataset_cnn_10k_cleaned.csv                   baris= 9,630 | isi kosong=  0.0% | tanggal terbaca=  0.0%
tempo.co           dataset_tempo_6k_cleaned.csv                  baris= 6,592 | isi kosong=  0.0% | tanggal terbaca=  0.0%
kompas.com         dataset_kompas_4k_cleaned.csv                 baris= 4,750 | isi kosong=  0.6% | tanggal terbaca=  0.0%
[1_raw                       ] total= 50,697 | per label={0: 20972, 1: 29725}  


# 3

In [3]:
# ---------------- 2. BENTUK TEKS (aturan sama untuk semua kelas) -------------
raw["text"] = [tp.build_text(t, b) for t, b in zip(raw.title, raw.body)]
raw["n_words_raw"] = raw.text.str.split().str.len()

repeat = pd.Series([bool(tp._alnum(b)) and tp._alnum(b).startswith(tp._alnum(tp.clean(t))) and bool(tp._alnum(t))
                    for t, b in zip(raw.title, raw.body.map(tp.clean))], index=raw.index)
print("\nBody diawali judul (judul diulang di sumber), per sumber:")
print(raw.assign(rep=repeat).groupby("source")["rep"].mean().mul(100).round(1).to_string())

df = raw[raw.text.str.len() > 0].copy()
log("2_teks_terbentuk", df)
df = df[df.n_words_raw >= CFG["MIN_WORDS"]].copy()
log("2b_min_kata", df, f">= {CFG['MIN_WORDS']} kata")


Body diawali judul (judul diulang di sumber), per sumber:
source
cnnindonesia.com    100.0
hoax_archive_15k      0.4
komdigi.go.id         0.0
kompas.com           99.4
tempo.co            100.0
turnbackhoax.id       4.1
[2_teks_terbentuk            ] total= 50,675 | per label={0: 20951, 1: 29724}  
[2b_min_kata                 ] total= 45,432 | per label={0: 20930, 1: 24502}  >= 10 kata


# 4

In [4]:
# ---------------- 3. SUB-TOPIK: word-boundary (bukan substring) --------------
SUB_TOPIC_RULES = {
    "pemilu": ["pemilu", "pilpres", "pilkada", "kpu", "bawaslu", "surat suara", "sirekap", "coblos", "capres", "cawapres", "cawagub", "caleg", "debat", "partai", "pdip", "gerindra", "golkar", "nasdem", "pks", "demokrat", "kpud", "panwaslu", "koalisi", "oposisi", "cawe-cawe", "dinasti", "legislatif"],
    "kebijakan": ["bansos", "subsidi", "bbm", "ikn", "apbn", "apbd", "pajak", "omnibus", "undang-undang", "perpres", "perda", "tarif listrik", "bpjs", "kebijakan pemerintah", "kementerian", "pemerintah", "negara", "asn", "bumn", "pertamina", "bulog", "beras bansos", "makan siang", "kemensos", "kemenkeu", "kemendagri", "reshuffle", "kabinet", "istana negara", "anggaran"],
    "tokoh": ["jokowi", "prabowo", "gibran", "anies", "ganjar", "megawati", "menteri", "presiden", "pejabat", "mahfud", "gubernur", "bupati", "walikota", "dpr", "mpr", "dprd", "puan", "erick thohir", "bahlil", "kaesang", "ridwan kamil", "ahok", "khofifah", "kades", "sri mulyani", "luhut", "nadiem", "basuki", "tito karnavian", "yasonna", "paspampres"],
    "hukum": ["korupsi", "kpk", "mk", "mahkamah", "polri", "kejaksaan", "tangkap", "ott", "sidang", "vonis", "hakim", "polisi", "tni", "aparat", "kejagung", "jaksa", "kasasi", "mahkamah agung", "yudikatif"],
}
NOISE = ["hujan es", "imunitas", "kanker", "diabetes", "ramuan", "vaksin hpv", "stroke", "gempa bumi", "tsunami", "cuaca ekstrem",
         "pelatih timnas", "liga 1", "piala aff", "frank de boer", "ronaldo", "messi", "shin tae-yong", "pesulap", "zodiak",
         "air kelapa", "janin", "kehamilan", "rahim", "kolesterol", "asam urat"]
wb = lambda words: re.compile(r"(?<!\w)(?:" + "|".join(re.escape(w) for w in words) + r")(?!\w)", re.I)
RULES_WB = {k: wb(v) for k, v in SUB_TOPIC_RULES.items()}; NOISE_WB = wb(NOISE)

def topic_wb(t):
    if NOISE_WB.search(t): return "bukan_politik"
    for k, rx in RULES_WB.items():
        if rx.search(t): return k
    return "umum"
def topic_substring(t):                                   # perilaku LAMA (hanya untuk perbandingan)
    t = t.lower()
    if any(w in t for w in NOISE): return "bukan_politik"
    for k, ws in SUB_TOPIC_RULES.items():
        if any(w in t for w in ws): return k
    return "umum"

df["sub_topic"] = df.text.map(topic_wb)
old = df.text.map(topic_substring)
changed = (old != df.sub_topic)
print(f"\nTopik berubah bila substring -> word-boundary: {changed.sum():,} teks ({changed.mean()*100:.1f}%). Contoh pemicu substring:")
print(df[changed].assign(lama=old[changed]).groupby(["lama", "sub_topic"]).size().sort_values(ascending=False).head(6).to_string())

df = df[~df.sub_topic.isin(["umum", "bukan_politik"])].copy()
log("3_filter_politik", df)


Topik berubah bila substring -> word-boundary: 4,682 teks (10.3%). Contoh pemicu substring:
lama       sub_topic
kebijakan  umum         1143
hukum      umum         1142
kebijakan  tokoh         798
tokoh      umum          699
pemilu     kebijakan     209
kebijakan  hukum         181
[3_filter_politik            ] total= 31,479 | per label={0: 20224, 1: 11255}  


# 5: Deduplikasi & Graf Kemiripan Topik (Near-Duplicates & Connected Components)

In [5]:
# ---------------- 4. BATAS PANJANG SAMA + DEDUP + GRUP ----------------------
df["text"] = df.text.map(lambda t: tp.cap_sentences(t, CFG["MAX_WORDS"]))
df["n_words"] = df.text.str.split().str.len()
df = df[df.n_words >= CFG["MIN_WORDS"]].reset_index(drop=True)
log("4_batas_panjang_sama", df, f"maks {CFG['MAX_WORDS']} kata, semua label")

before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
log("4b_duplikat_persis", df, f"dibuang {before-len(df):,}")

def similarity_edges(texts, thr, chunk=500):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform(texts).tocsr(); XT = X.T.tocsr()
    I, J, V = [np.empty(0, int)], [np.empty(0, int)], [np.empty(0, np.float32)]
    for s in range(0, X.shape[0], chunk):
        S = (X[s:s + chunk] @ XT).tocoo()
        m = (S.data >= thr) & (S.row + s < S.col)
        I.append(S.row[m] + s); J.append(S.col[m]); V.append(S.data[m])
    return np.concatenate(I), np.concatenate(J), np.concatenate(V)

def comp(n, I, J):
    return connected_components(coo_matrix((np.ones(len(I)), (I, J)), shape=(n, n)), directed=False)[1]

n = len(df); lab = df.label.to_numpy()
I, J, V = similarity_edges(df.text.tolist(), CFG["GROUP_COS"])
print(f"\nPasangan mirip (cos >= {CFG['GROUP_COS']}): {len(I):,}")

hi = V >= CFG["NEAR_DUP_COS"]
keep = np.ones(n, bool)
cross = hi & (lab[I] != lab[J])                          # mirip tapi beda label -> ambigu: buang keduanya
keep[I[cross]] = False; keep[J[cross]] = False
same = hi & (lab[I] == lab[J])
if same.any():
    c = comp(n, I[same], J[same]); first = pd.Series(np.arange(n)).groupby(c).transform("min").to_numpy()
    touched = np.zeros(n, bool); touched[I[same]] = True; touched[J[same]] = True
    keep &= ~(touched & (np.arange(n) != first))
print(f"Near-duplicate cos>={CFG['NEAR_DUP_COS']}: beda-label dibuang={int(cross.sum()):,} pasangan | sama-label dibuang={int((~keep).sum()):,} baris total")

old_idx = np.flatnonzero(keep); remap = -np.ones(n, int); remap[old_idx] = np.arange(len(old_idx))
m = keep[I] & keep[J]
df = df.loc[keep].reset_index(drop=True)
g = comp(len(df), remap[I[m]], remap[J[m]])
df["group_id"] = g
sizes = pd.Series(g).value_counts()
print(f"Grup near-duplicate: {len(sizes):,} grup | terbesar={sizes.iloc[0]} | grup berukuran >1: {(sizes>1).sum():,}")
log("4c_dedup_near", df)

[4_batas_panjang_sama        ] total= 31,479 | per label={0: 20224, 1: 11255}  maks 60 kata, semua label
[4b_duplikat_persis          ] total= 31,461 | per label={0: 20224, 1: 11237}  dibuang 18

Pasangan mirip (cos >= 0.7): 2,070
Near-duplicate cos>=0.9: beda-label dibuang=0 pasangan | sama-label dibuang=433 baris total
Grup near-duplicate: 29,651 grup | terbesar=8 | grup berukuran >1: 1,213
[4c_dedup_near               ] total= 31,028 | per label={0: 20162, 1: 10866}  


# 6: Penyeimbangan Terkendali (Controlled Balancing) & Stratified Group Split

In [6]:
# ---------------- 5. AUDIT SEBELUM PENYEIMBANGAN (tidak ada filter di sini) ---
def audit(d, title):
    print("\n" + "=" * 78); print(title); print("=" * 78)
    a = d.assign(dup=d.text.map(tp.dup_title), endp=d.text.str.endswith((".", "!", "?")),
                 w=d.text.str.split().str.len()).groupby("label")
    print(pd.DataFrame({"n": a.size(), "judul_ganda_%": a.dup.mean()*100, "berakhir_tanda_%": a.endp.mean()*100,
                        "kata_mean": a.w.mean(), "kata_sd": a.w.std(), "kata_median": a.w.median(), "kata_max": a.w.max()}).round(2).to_string())
    print("\nsub_topik x label:"); print(pd.crosstab(d.sub_topic, d.label).to_string())
    print("\nsumber x label:");   print(pd.crosstab(d.source, d.label).to_string())
    if d.date.notna().any():
        print("\ntahun x label (dari kolom tanggal, hanya yang terbaca):")
        print(pd.crosstab(d.date.dt.year, d.label).to_string())
    print("\nKata pembuka teratas per label:")
    for l in (0, 1):
        print(f"  label {l}:", d[d.label == l].text.str.split().str[0].str.lower().value_counts().head(8).to_dict())

audit(df, "AUDIT SEBELUM PENYEIMBANGAN")


AUDIT SEBELUM PENYEIMBANGAN
           n  judul_ganda_%  berakhir_tanda_%  kata_mean  kata_sd  kata_median  kata_max
label                                                                                   
0      20162           3.82             100.0      48.96     9.78         51.0        60
1      10866          11.14             100.0      42.57    14.84         46.0        60

sub_topik x label:
label          0     1
sub_topic             
hukum        224  1169
kebijakan   3372  3561
pemilu     14894  2354
tokoh       1672  3782

sumber x label:
label                0     1
source                      
cnnindonesia.com  9483     0
hoax_archive_15k     0  7369
komdigi.go.id        0   450
kompas.com        4453     0
tempo.co          6226     0
turnbackhoax.id      0  3047

tahun x label (dari kolom tanggal, hanya yang terbaca):
label     1
date       
2015.0    5
2016.0   26
2017.0  104
2018.0  172
2019.0  221
2020.0  270
2021.0  171
2022.0  171
2023.0  338
2024.0  434
2025.0 

# 7

In [7]:
# ---------------- 6. PENYEIMBANGAN TERKENDALI + SPLIT BERKELOMPOK -----------
df["len_bin"] = pd.cut(df.n_words, bins=CFG["LEN_BINS"]).astype(str)

def coarsened_match(d, keys, seed):
    parts = []
    for _, gk in d.groupby(keys, observed=True):
        c = gk.label.value_counts()
        if len(c) < 2: continue
        k = int(c.min())
        parts += [gg.sample(k, random_state=seed) for _, gg in gk.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

bal = coarsened_match(df, ["sub_topic", "len_bin"], SEED)
log("6_seimbang_topik_panjang", bal, "strata = sub_topic x len_bin")
print(f"Ukuran: {len(df):,} -> {len(bal):,} (kehilangan {(1-len(bal)/len(df))*100:.1f}% demi kesetaraan topik & panjang)")

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=SEED)
fold = np.full(len(bal), -1)
for k, (_, te_idx) in enumerate(sgkf.split(bal, bal.label, bal.group_id)):
    fold[te_idx] = k
edges = np.cumsum([CFG["TRAIN_FOLDS"], CFG["VAL_FOLDS"], CFG["DEVTEST_FOLDS"]])
bal["split"] = np.select([fold < edges[0], fold < edges[1], fold < edges[2]], ["train", "val", "dev_test"], "lockbox")
bal["id"] = [f"POL2-{i+1:06d}" for i in range(len(bal))]
print(pd.crosstab(bal.split, bal.label).assign(total=lambda x: x.sum(axis=1)).to_string())
print("\nLOCKBOX disegel: hanya jumlahnya yang dilihat. Isi/statistiknya TIDAK dilaporkan sampai protokol selesai.")

[6_seimbang_topik_panjang    ] total= 14,468 | per label={0: 7234, 1: 7234}  strata = sub_topic x len_bin
Ukuran: 31,028 -> 14,468 (kehilangan 53.4% demi kesetaraan topik & panjang)
label        0     1  total
split                      
dev_test   723   723   1446
lockbox    722   724   1446
train     5066  5064  10130
val        723   723   1446

LOCKBOX disegel: hanya jumlahnya yang dilihat. Isi/statistiknya TIDAK dilaporkan sampai protokol selesai.


# 8

In [8]:
# ---------------- 7. VERIFIKASI TERHADAP KRITERIA YANG DITETAPKAN DI AWAL ----
res = []
dev = bal[bal.split != "lockbox"]                      # lockbox tidak ikut diaudit
tr_, va_, te_ = (bal[bal.split == s] for s in ("train", "val", "dev_test"))

for s, d in (("train", tr_), ("val", va_), ("dev_test", te_)):
    r = d.groupby("label").text.apply(lambda x: x.map(tp.dup_title).mean() * 100)
    res.append((f"C1 judul ganda [{s}]", abs(r[0] - r[1]), 5, abs(r[0] - r[1]) <= 5))
ks = ks_2samp(dev[dev.label == 0].n_words, dev[dev.label == 1].n_words).statistic
res.append(("C2 KS panjang kata", ks, 0.10, ks <= 0.10))
e = dev.groupby("label").text.apply(lambda x: x.str.endswith((".", "!", "?")).mean() * 100)
res.append(("C3 berakhir tanda baca", abs(e[0] - e[1]), 5, abs(e[0] - e[1]) <= 5))

def lenfeat(d):
    t = d.text; w = t.str.split().str.len()
    return np.c_[w, t.str.len(), t.str.count(r"[.!?]"), t.str.count(",")]
dt = DecisionTreeClassifier(max_depth=4, random_state=0).fit(lenfeat(tr_), tr_.label)
acc_len = accuracy_score(te_.label, dt.predict(lenfeat(te_)))
res.append(("C4 akurasi fitur panjang (dev_test)", acc_len, 0.60, acc_len <= 0.60))

viol = bal.groupby("group_id").split.nunique().gt(1).sum()
res.append(("C5 grup melintasi split", int(viol), 0, viol == 0))
tp_ = dev.groupby("label").sub_topic.value_counts(normalize=True).unstack(0).fillna(0)
dtopic = (tp_[0] - tp_[1]).abs().max() * 100
res.append(("C6 selisih proporsi sub-topik", dtopic, 2, dtopic <= 2))

print(f"{'Kriteria':<36}{'Nilai':>10}{'Ambang':>9}  Hasil")
for name, v, th, ok in res:
    print(f"{name:<36}{v:>10.3f}{th:>9}  {'LULUS' if ok else 'GAGAL'}")

first8 = lambda d: d.text.str.split().str[:8].str.join(" ")
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
lr = LogisticRegression(max_iter=2000).fit(vec.fit_transform(first8(tr_)), tr_.label)
acc8 = accuracy_score(te_.label, lr.predict(vec.transform(first8(te_))))
print(f"\nINFO akurasi 8 kata pertama (TF-IDF+LR, dev_test): {acc8*100:.2f}%  <- bila tinggi, gaya penulisan/sumber masih membedakan kelas")
print("INFO sumber x label (ketimpangan sumber TIDAK dapat diperbaiki oleh kode; perlu sumber tambahan):")
print(pd.crosstab(bal.source, bal.label).to_string())

if dev.date.notna().any():
    yr = pd.crosstab(dev.date.dt.year, dev.label); yr = yr.div(yr.sum(0), axis=1)
    overlap = float(np.minimum(yr.get(0, 0), yr.get(1, 0)).sum()) * 100 if {0, 1} <= set(yr.columns) else 0.0
    print(f"INFO tumpang-tindih distribusi tahun antar label: {overlap:.1f}%  (100% = identik; 0% = periode terpisah total)")
    if overlap < 50: print("   >> PERINGATAN: periode waktu kedua kelas hampir tidak beririsan; model dapat memakai 'zaman' sebagai petunjuk label.")
audit(dev, "AUDIT SETELAH PENYEIMBANGAN (tanpa lockbox)")
ALL_OK = all(ok for *_, ok in res)
print("\nSEMUA KRITERIA LULUS" if ALL_OK else "\nADA KRITERIA GAGAL -> perbaiki pipeline, JANGAN longgarkan ambang tanpa alasan tertulis.")

Kriteria                                 Nilai   Ambang  Hasil
C1 judul ganda [train]                   7.604        5  GAGAL
C1 judul ganda [val]                     8.714        5  GAGAL
C1 judul ganda [dev_test]                7.884        5  GAGAL
C2 KS panjang kata                       0.051      0.1  LULUS
C3 berakhir tanda baca                   0.000        5  LULUS
C4 akurasi fitur panjang (dev_test)      0.666      0.6  GAGAL
C5 grup melintasi split                  0.000        0  LULUS
C6 selisih proporsi sub-topik            0.182        2  LULUS

INFO akurasi 8 kata pertama (TF-IDF+LR, dev_test): 88.52%  <- bila tinggi, gaya penulisan/sumber masih membedakan kelas
INFO sumber x label (ketimpangan sumber TIDAK dapat diperbaiki oleh kode; perlu sumber tambahan):
label                0     1
source                      
cnnindonesia.com  2779     0
hoax_archive_15k     0  5385
komdigi.go.id        0   105
kompas.com        1443     0
tempo.co          3012     0
turnbackhoa

C:\Users\ASUS\AppData\Local\Temp\ipykernel_2676\4138283280.py:40: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  yr = pd.crosstab(dev.date.dt.year, dev.label); yr = yr.div(yr.sum(0), axis=1)


          n  judul_ganda_%  berakhir_tanda_%  kata_mean  kata_sd  kata_median  kata_max
label                                                                                  
0      6512           3.76             100.0      46.52    11.62         50.0        60
1      6510          11.52             100.0      46.46    12.31         50.0        60

sub_topik x label:
label         0     1
sub_topic            
hukum       207   199
kebijakan  2680  2691
pemilu     2132  2126
tokoh      1493  1494

sumber x label:
label                0     1
source                      
cnnindonesia.com  2500     0
hoax_archive_15k     0  4841
komdigi.go.id        0    97
kompas.com        1292     0
tempo.co          2720     0
turnbackhoax.id      0  1572

tahun x label (dari kolom tanggal, hanya yang terbaca):
label     1
date       
2015.0    3
2016.0   19
2017.0   72
2018.0  118
2019.0  144
2020.0  158
2021.0  107
2022.0  112
2023.0  223
2024.0  277
2025.0   36
2026.0   28

Kata pembuka teratas 

# 9

In [9]:
# ---------------- 8. SIMPAN (berkas baru; tidak menimpa data lama) + MANIFEST -
cols = ["id", "text", "label", "sub_topic", "source", "group_id", "date"]

# 1. Simpan Split Utama
for sp in ("train", "val", "dev_test"):
    bal[bal.split == sp][cols].to_csv(os.path.join(OUT_DIR, f"{sp}.csv"), index=False, encoding="utf-8-sig")

bal[bal.split != "lockbox"][cols + ["split"]].to_csv(os.path.join(OUT_DIR, "dataset_v2_full.csv"), index=False, encoding="utf-8-sig")

# 2. Segel Lockbox
LOCK_DIR = os.path.join(OUT_DIR, "LOCKBOX")
os.makedirs(LOCK_DIR, exist_ok=True)
bal[bal.split == "lockbox"][cols].to_csv(os.path.join(LOCK_DIR, "lockbox.csv"), index=False, encoding="utf-8-sig")

with open(os.path.join(LOCK_DIR, "JANGAN_DIBUKA.txt"), "w", encoding="utf-8") as f:
    f.write(
        "Set uji tersegel. Dibuka SATU KALI untuk pelaporan akhir, setelah model, ambang, dan analisis dikunci di PROTOKOL.\n"
        "Tidak boleh dipakai untuk tuning, pemilihan model, atau diagnosis. Catat pembukaan di PROTOKOL (tanggal + hash).\n"
    )

# 3. Simpan Log Audit & Kriteria
pd.DataFrame(LOG).to_csv(os.path.join(OUT_DIR, "audit_log_per_langkah.csv"), index=False)
pd.DataFrame([(n_, v, th, ok) for n_, v, th, ok in res], columns=["kriteria", "nilai", "ambang", "lulus"]).to_csv(
    os.path.join(OUT_DIR, "hasil_kriteria.csv"), index=False
)

# 4. Fungsi Hashing SHA-256
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): 
            h.update(ch)
    return h.hexdigest()

# 5. Simpan Konfigurasi
cfg_out = dict(
    CFG, 
    python=platform.python_version(), 
    pandas=pd.__version__, 
    numpy=np.__version__,
    created=datetime.datetime.now().isoformat(timespec="seconds"), 
    raw_dir=RAW_DIR, 
    all_criteria_passed=bool(ALL_OK)
)
with open(os.path.join(OUT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg_out, f, indent=2)

# 6. Buat Manifes Checksum SHA-256
man = [{"berkas": f, "sha256": sha(os.path.join(OUT_DIR, f))} for f in sorted(os.listdir(OUT_DIR)) if f not in ("MANIFEST_sha256.csv", "LOCKBOX")]
man += [{"berkas": "RAW/" + f, "sha256": sha(os.path.join(RAW_DIR, f))} for f in sorted(df_f for df_f in raw["file"].unique())]
man += [{"berkas": "LOCKBOX/lockbox.csv", "sha256": sha(os.path.join(LOCK_DIR, "lockbox.csv"))}]

# --- PERBAIKAN: Deteksi lokasi riil text_pipeline.py secara dinamis ---
tp_file_path = getattr(tp, "__file__", None)
if not tp_file_path or not os.path.exists(tp_file_path):
    for candidate in ["text_pipeline.py", "../text_pipeline.py"]:
        if os.path.exists(candidate):
            tp_file_path = candidate
            break

if not tp_file_path or not os.path.exists(tp_file_path):
    raise FileNotFoundError("Berkas text_pipeline.py tidak ditemukan untuk pembuatan manifes SHA-256!")

man += [{"berkas": "text_pipeline.py", "sha256": sha(tp_file_path)}]
# --------------------------------------------------------------------

pd.DataFrame(man).to_csv(os.path.join(OUT_DIR, "MANIFEST_sha256.csv"), index=False)

print("=" * 80)
print("✓ SEMUA BERKAS BERHASIL DISIMPAN & MANIFEST SHA-256 DIBUAT DENGAN SUKSES!")
print("Tersimpan di:", os.path.abspath(OUT_DIR))
print("Daftar berkas:", sorted(os.listdir(OUT_DIR)))
print("=" * 80)

✓ SEMUA BERKAS BERHASIL DISIMPAN & MANIFEST SHA-256 DIBUAT DENGAN SUKSES!
Tersimpan di: e:\SEMESTER 5\TA\PROJECT\data\processed_v2
Daftar berkas: ['LOCKBOX', 'MANIFEST_sha256.csv', 'audit_log_per_langkah.csv', 'config.json', 'dataset_v2_full.csv', 'dev_test.csv', 'hasil_kriteria.csv', 'train.csv', 'val.csv']
